# Module 3a — Generate evaluation questions from your library

**Time:** ~10 min · **Output:** `workshop_outputs/03_questions.csv`, `workshop_outputs/03_answers.csv`

1. Read and chunk the same documents you uploaded to MyGPT
2. Ask an LLM to write one question + reference answer per sampled chunk
3. Add unanswerable questions
4. **Review the questions yourself**, then run them through MyGPT

In [ ]:
import sys
from pathlib import Path

# Make the repo root importable whether Jupyter was started from the root or this folder
ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from common.mygpt_client import MyGPTClient, outputs_dir, load_fallback_pubmedqa  # noqa: E402
OUT = outputs_dir()

In [ ]:
DOCS_DIR = ROOT / "data" / "BioASQ" / "pdfs"
N_CHUNKS = 20                              # number of questions to generate (reduce on slow laptops)
CHUNK_SIZE = 1500                          # characters per chunk used for question generation
SEED = 42

client = MyGPTClient.from_env()
print("Generator LLM:", client.llm, "via", client.ollama_url)

## 1. Read and chunk documents

In [ ]:
def read_text(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return path.read_text(encoding="utf-8", errors="ignore")
    if suffix == ".pdf":
        from pypdf import PdfReader
        return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
    if suffix == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(path).paragraphs)
    return ""

chunks = []
for path in sorted(DOCS_DIR.glob("*")):
    text = re.sub(r"\s+", " ", read_text(path)).strip()
    for start in range(0, len(text), CHUNK_SIZE):
        piece = text[start:start + CHUNK_SIZE]
        if len(piece) > 400:                      # skip tiny tail chunks
            chunks.append({"source_doc": path.name, "chunk": piece})

print(len(chunks), "chunks from", len({c["source_doc"] for c in chunks}), "documents")
random.seed(SEED)
sample = random.sample(chunks, min(N_CHUNKS, len(chunks)))

## 2. Generate question + reference answer per chunk
The prompt asks for **specific, self-contained** questions: a reader shouldn't need to see the chunk to understand the question.

In [ ]:
QGEN_PROMPT = """You are building a benchmark for a question-answering system over a document library.
Read the passage and write ONE question that:
- can be answered from this passage alone,
- is specific and self-contained (do not say "the passage", "this document", "the author"),
- a researcher in this field would realistically ask.
Then write a concise reference answer (1-2 sentences) using only the passage.

Return JSON: {{"question": "...", "answer": "..."}}

PASSAGE:
{chunk}
"""

rows = []
for i, item in enumerate(sample, 1):
    try:
        raw = client.ollama(QGEN_PROMPT.format(chunk=item["chunk"]), temperature=0.2, json_mode=True)
        qa = json.loads(raw)
        rows.append({"question": qa["question"].strip(), "reference_answer": qa["answer"].strip(),
                     "answerable": 1, "source_doc": item["source_doc"], "source_chunk": item["chunk"]})
        print(f"[{i}] {qa['question']}")
    except Exception as exc:
        print(f"[{i}] skipped: {exc}")

## 3. Add unanswerable questions
Write questions that look plausible for your field but that your documents **do not** answer. You need these to test whether HI catches hallucinations.

In [ ]:
UNANSWERABLE = [
    "What is the boiling point of tungsten?",
    "Who won the 2018 FIFA World Cup?",
    "What dose of imatinib is recommended for adults with chronic myeloid leukemia?",   # ← replace with near-miss questions for YOUR domain
]
NOT_IN_LIBRARY = "The library does not contain this information."
rows += [{"question": q, "reference_answer": NOT_IN_LIBRARY, "answerable": 0,
          "source_doc": "", "source_chunk": ""} for q in UNANSWERABLE]

questions = pd.DataFrame(rows)
questions.insert(0, "question_id", range(1, len(questions) + 1))
questions.to_csv(OUT / "03_questions.csv", index=False)
questions[["question_id", "question", "reference_answer", "answerable"]]

## 4. ✋ Human review (do not skip)
Open `workshop_outputs/03_questions.csv` in a spreadsheet or the Jupyter CSV viewer:
- **delete** questions that are vague, trivial or wrong
- **edit** reference answers that are incomplete
- check that each question really is answerable from its `source_chunk`

Save the file, then run the next cell to reload it.

In [ ]:
questions = pd.read_csv(OUT / "03_questions.csv")
print(len(questions), "questions after review;", int((questions.answerable == 0).sum()), "unanswerable")

## 5. Run every question through MyGPT

In [ ]:
answers = []
for i, r in questions.iterrows():
    print(f"[{i + 1}/{len(questions)}] {r.question}")
    try:
        res = client.ask(r.question)
    except Exception as exc:
        print("   failed:", exc)
        res = {"error": str(exc)}
    answers.append({**r.to_dict(), **{k: v for k, v in res.items() if k != "question"}})

answers = pd.DataFrame(answers)
answers.to_csv(OUT / "03_answers.csv", index=False)
answers[["question_id", "question", "answer", "QRS", "ARS", "HI"]].head(10)

➡️ Next: **03b_label_and_judge.ipynb**